# Inside middleware
You can access runtime information in middleware to create dynamic prompts, modify messages, or control agent behavior based on user context.
Use the Runtime parameter to access the Runtime object inside node-style hooks. For wrap-style hooks, the Runtime object is available inside the ModelRequest parameter.

In [2]:
import os

from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from pprint import pprint
from langchain.tools import tool

from langchain.chat_models import init_chat_model
from rich import print as rprint

In [4]:
model_free = init_chat_model("openai/gpt-oss-20b",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=10000, temperature=0.0)

model_basic = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)

model_medium = init_chat_model("openai/gpt-5.6-luna",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)

model_advanced = init_chat_model("openai/gpt-5.6-luna-pro",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)

model_safety = init_chat_model("nvidia/nemotron-3.5-content-safety:free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=10000, temperature=0.0)



In [5]:
from dataclasses import dataclass

from langchain.messages import AnyMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import dynamic_prompt, ModelRequest, before_model, after_model
from langgraph.runtime import Runtime


@dataclass
class Context:
    user_name: str

# Dynamic prompts
@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    user_name = request.runtime.context.user_name  
    system_prompt = f"You are a helpful assistant. Address the user as {user_name}."
    return system_prompt

# Before model hook
@before_model
def log_before_model(state: AgentState, runtime: Runtime[Context]) -> dict | None:
    print(f"Processing request for user: {runtime.context.user_name}")
    return None

# After model hook
@after_model
def log_after_model(state: AgentState, runtime: Runtime[Context]) -> dict | None:
    print(f"Completed request for user: {runtime.context.user_name}")
    return None


In [7]:

agent = create_agent(
    model=model_advanced,
    tools=[],
    middleware=[dynamic_system_prompt, log_before_model, log_after_model],
    context_schema=Context
)


In [8]:

agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    context=Context(user_name="John Smith")
)

Processing request for user: John Smith
Completed request for user: John Smith


{'messages': [HumanMessage(content="What's my name?", additional_kwargs={}, response_metadata={}, id='7044820e-dadd-48e4-af50-a86068f1e0ca'),
  AIMessage(content='Your name is John Smith.', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-5.6-luna-pro', 'id': 'gen-1787373955-x512FNIF31hzaeMVtKYR', 'created': 1787373955, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0004576, 'cost_details': {'upstream_inference_completions_cost': 0.0001368, 'upstream_inference_prompt_cost': 0.0003208, 'upstream_inference_cost': 0.0004576}}, id='lc_run--01a027ca-2780-7700-a5c4-8ff5b640fa5a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1604, 'output_tokens': 114, 'total_tokens': 1718, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 67}})]}